# ML-02 — Research Question and Provisional Lane (Freestyle Direction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames my **Freestyle Research Project** for the FlyRank Machine Learning Internship. As an aspiring Data Science / ML Specialist entering the industry job market, this project is designed to tackle a cutting-edge, real-world machine learning challenge: **Generative Engine Optimization (GEO) & AI Referral Traffic Intelligence** combined with multi-signal content performance modeling.

## 1. My lane (or freestyle) and why

**Lane Declaration:** **Freestyle Direction** — *AI Referral & Generative Engine Optimization (GEO) Opportunity Scoring: Multi-Signal Machine Learning for AI Search Traffic & Content Trajectories.*

**One-Paragraph Plan & Motivation:**
As an aspiring Data Science and Machine Learning professional, my goal during this 8-week internship is to build an industry-grade, resume-defining portfolio piece. Rather than restricting my work to standard SEO rules, I am pursuing a freestyle direction at the intersection of **AI Search Assistants** (ChatGPT, Perplexity, Claude, Gemini) and traditional organic performance. As user search behavior shifts toward conversational AI, web publishers face a dual challenge: protecting decaying organic search traffic while capturing sparse but high-intent AI referral traffic. By leveraging both the 30k starter dataset and the 78M+ warehouse dataset, I will build an end-to-end ML pipeline that scores content for **AI Visibility Gaps** (high search demand but under-indexed AI traffic) and predicts content performance trajectories using client-holdout cross-validation.

In [2]:
# Verification of Freestyle project parameters and dataset access
from pathlib import Path
import os

raw_path = Path('../../data/raw/content_refresh_anonymized.csv')
if not raw_path.exists():
    raw_path = Path('data/raw/content_refresh_anonymized.csv')

print('[✓] Project Mode: FREESTYLE (AI Referral & GEO Opportunity Scoring)')
print(f'[✓] Primary Data Source: {raw_path.resolve()}')
print(f'[✓] Dataset Exists & Ready: {raw_path.exists()}')


[✓] Project Mode: FREESTYLE (AI Referral & GEO Opportunity Scoring)
[✓] Primary Data Source: D:\Flyrank\FlyRank-Machine-Learning-Internship\data\raw\content_refresh_anonymized.csv
[✓] Dataset Exists & Ready: True


## 2. The question: decision, action, cost of a wrong call

### Problem Framing & Core Questions
- **Research Question:** *"Which high-demand content items exhibit an 'AI Visibility Gap' (strong search impressions but disproportionately low AI referral traffic), and how can machine learning models prioritize these pages for Generative Engine Optimization (GEO) and content decay protection?"*
- **Unit of Analysis:** A single pseudonymized content item (`content_id`) evaluated over a trailing 90-day window.
- **Decision Improved:** Deciding which specific articles an editorial and growth team should optimize for AI conversational engine discovery vs. standard organic search refresh.
- **Who Acts & Action Taken:** Content Editors, SEO Strategists, and AI Growth Engineers. Guided by ML scores and transparent reason codes (`ai_visibility_gap`, `high_demand_low_ai_referral`, `stale_visible_page`), they execute targeted interventions:
  - Structuring content with clear entity definitions, bulleted summaries, and FAQ schema for LLM retrieval (`ai_visibility_gap`)
  - Rewriting meta tags and updating outdated figures (`stale_visible_page` / `low_ctr_visible_page`)
  - Expanding thin articles with high organic/AI upside (`thin_visible_page`)
- **Cost of a Wrong Call:**
  - **False Positive (flagging a page with zero AI potential):** Wastes 3–5 hours of editorial/engineering time restructuring content that AI engines will not cite.
  - **False Negative (missing a high-potential AI opportunity):** Loss of market share and brand referral traffic as users transition from traditional Google searches to AI engine queries.
- **Why Data & ML Help:** AI-referred traffic is sparse (~6.4% of total pages) and exhibits complex non-linear relationships with search volume, position, word count, and engagement rate. Traditional heuristic rules fail on sparse signals, whereas machine learning models (e.g., Random Forest, Gradient Boosting with class weighting) effectively isolate high-probability candidates from background noise.

In [3]:
# Formal definition of Freestyle Decision Matrix for ML Resume Portfolio
freestyle_matrix = {
    'Target Persona': 'Growth Lead / AI SEO Strategist',
    'Unit of Analysis': 'content_id (pseudonymized content item)',
    'Primary ML Task': 'Imbalanced Classification & Priority Ranking (Precision@K)',
    'Key Target Signals': 'ai_sessions_90d, ai_traffic_pct, trend_direction',
    'Primary Action': 'Generative Engine Optimization (GEO) & Content Refresh'
}

for k, v in freestyle_matrix.items():
    print(f'{k:24s}: {v}')


Target Persona          : Growth Lead / AI SEO Strategist
Unit of Analysis        : content_id (pseudonymized content item)
Primary ML Task         : Imbalanced Classification & Priority Ranking (Precision@K)
Key Target Signals      : ai_sessions_90d, ai_traffic_pct, trend_direction
Primary Action          : Generative Engine Optimization (GEO) & Content Refresh


## 3. Quick look at the data (2-3 real numbers)

Exploration of the starter dataset (`data/raw/content_refresh_anonymized.csv`) provides four strong empirical numbers supporting this freestyle direction:

1. **AI Traffic Sparsity & Impact:** Out of **30,000 content items**, **1,930 items (6.43%)** receive direct AI-referred traffic (`ai_sessions_90d > 0`), generating a total of **6,135 AI sessions**. Notably, when AI traffic is present on a page, it accounts for an average of **11.94%** of that page's total session volume — demonstrating high traffic impact.
2. **The AI Visibility Gap:** **16,726 items (55.75%)** generate high organic search demand (`impressions_90d >= 500`), but **15,033 of these high-demand items (89.88%)** receive **zero AI referral traffic** (`ai_sessions_90d == 0`). This reveals a massive "AI Visibility Gap" ready for machine learning prioritization.
3. **Dual Decay Challenge:** **16,262 items (54.21%)** suffer from downward organic trends (`trend_direction == "down"`), confirming that content decay and AI discovery must be addressed simultaneously.
4. **ML vs. Heuristic Rule Performance:** Client-holdout model benchmarks show that standard baseline rules achieve a **Precision@50 of 0.240**, while a Random Forest model achieves **0.680** — delivering a **2.83x performance lift** in identifying critical candidate pages.

In [4]:
import pandas as pd
import json
from pathlib import Path

# 1. Load starter raw dataset
data_path = Path('../../data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
    data_path = Path('data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(data_path)

total_rows = len(df)
ai_active = (df['ai_sessions_90d'] > 0).sum()
ai_active_pct = (ai_active / total_rows) * 100
total_ai_sessions = df['ai_sessions_90d'].sum()
avg_ai_share = df[df['ai_sessions_90d'] > 0]['ai_traffic_pct'].mean()

high_demand = (df['impressions_90d'] >= 500).sum()
high_demand_zero_ai = ((df['impressions_90d'] >= 500) & (df['ai_sessions_90d'] == 0)).sum()
zero_ai_pct_in_demand = (high_demand_zero_ai / high_demand) * 100

declining_count = (df['trend_direction'] == 'down').sum()
declining_pct = (declining_count / total_rows) * 100

print('=' * 70)
print('FREESTYLE EMPIRICAL EVIDENCE (AI REFERRALS & CONTENT DECAY)')
print('=' * 70)
print(f'1. Total Inventory Analyzed    : {total_rows:,} content items across {df["client_id"].nunique()} clients')
print(f'2. AI-Referred Content Pool    : {ai_active:,} items ({ai_active_pct:.2f}%) | Total AI Sessions: {total_ai_sessions:,}')
print(f'   -> Mean AI Traffic Share   : {avg_ai_share:.2f}% of sessions (when AI traffic exists)')
print(f'3. AI Visibility Gap Pool      : {high_demand_zero_ai:,} out of {high_demand:,} high-demand pages ({zero_ai_pct_in_demand:.2f}%) have 0 AI sessions')
print(f'4. Organic Decay Rate          : {declining_count:,} items ({declining_pct:.2f}%) in downward trend')

# 2. Load model benchmark results
results_path = Path('../../outputs/model_results.json')
if not results_path.exists():
    results_path = Path('outputs/model_results.json')

if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print('\n' + '=' * 70)
    print('MODEL BENCHMARK RESULTS (Client-Holdout Split)')
    print('=' * 70)
    if 'baseline' in results:
        b_metrics = results['baseline']
        print(f" - {'baseline_rules':22s}: Precision@50 = {b_metrics.get('baseline_precision_at_50', 0):.3f} | ROC-AUC = {b_metrics.get('baseline_roc_auc', 0):.3f}")
    if 'models' in results:
        for model_name, metrics in results['models'].items():
            prec50 = metrics.get('precision_at_50', 0)
            roc_auc = metrics.get('roc_auc', 0)
            print(f" - {model_name:22s}: Precision@50 = {prec50:.3f} | ROC-AUC = {roc_auc:.3f}")


FREESTYLE EMPIRICAL EVIDENCE (AI REFERRALS & CONTENT DECAY)
1. Total Inventory Analyzed    : 30,000 content items across 32 clients
2. AI-Referred Content Pool    : 1,930 items (6.43%) | Total AI Sessions: 6,135
   -> Mean AI Traffic Share   : 11.94% of sessions (when AI traffic exists)
3. AI Visibility Gap Pool      : 15,033 out of 16,726 high-demand pages (89.88%) have 0 AI sessions
4. Organic Decay Rate          : 16,262 items (54.21%) in downward trend

MODEL BENCHMARK RESULTS (Client-Holdout Split)
 - baseline_rules        : Precision@50 = 0.240 | ROC-AUC = 0.627
 - decision_tree         : Precision@50 = 0.620 | ROC-AUC = 0.742
 - logistic_regression   : Precision@50 = 0.400 | ROC-AUC = 0.700
 - random_forest         : Precision@50 = 0.680 | ROC-AUC = 0.747


## 4. Careful words: what I can and can't claim

### What I CAN claim:
- **Observed Correlations:** Empirical relationships between web performance metrics (search impressions, CTR, position, word count) and AI referral session occurrences (`sessions_ai`).
- **Relative Prioritization:** A data-driven Machine Learning ranking that identifies high-demand pages with disproportionately low AI traffic for targeted review.
- **Decision Support for GEO:** Prioritizing editorial efforts toward content structuring, entity clarity, and FAQ addition to improve AI engine indexability.

### What I CANNOT claim:
- **LLM Internal Decoding:** That this project inspects or reverse-engineers the internal weights or prompt retrieval mechanisms of ChatGPT, Perplexity, or Gemini.
- **Guaranteed LLM Citation:** That optimizing a page for GEO *guarantees* that an AI conversational model will cite or link to the page.
- **Causal Proof:** That content edits *caused* an increase in AI traffic without running a formal controlled experiment.

In [5]:
# Claim boundaries verification for scientific & professional integrity
allowed_claims = [
    'Observed historical correlations between GSC/GA4 signals and AI traffic',
    'Relative opportunity scoring for Generative Engine Optimization (GEO)',
    'Decision-support to prioritize editor time for AI search readiness'
]

forbidden_claims = [
    'Reverse engineering LLM internal retrieval mechanisms',
    'Guaranteed LLM citations or rank #1 AI responses',
    'Causal attribution without controlled A/B testing'
]

print('[✓] SCIENTIFICALLY SOUND CLAIMS:')
for claim in allowed_claims:
    print(f'   - {claim}')

print('\n[X] FORBIDDEN OVER-CLAIMS:')
for claim in forbidden_claims:
    print(f'   - {claim}')


[✓] SCIENTIFICALLY SOUND CLAIMS:
   - Observed historical correlations between GSC/GA4 signals and AI traffic
   - Relative opportunity scoring for Generative Engine Optimization (GEO)
   - Decision-support to prioritize editor time for AI search readiness

[X] FORBIDDEN OVER-CLAIMS:
   - Reverse engineering LLM internal retrieval mechanisms
   - Guaranteed LLM citations or rank #1 AI responses
   - Causal attribution without controlled A/B testing


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.